In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm import tqdm

In [ ]:
# =============== Cấu hình cơ bản ===============
data_dir = "/kaggle/input/asl-alphabet/asl_split"
batch_size = 256
epochs = 70
lr = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device, "(", torch.cuda.device_count(), "GPUs )")

In [ ]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),  # dịch ngang + dọc ±5%
    transforms.RandomResizedCrop(224, scale=(0.95, 1.05)),        # zoom ±5%
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # Pixel ∈ [0,1] rồi chuẩn hóa
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [ ]:
train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

num_classes = len(train_dataset.classes)
print("Số lớp:", num_classes)

In [ ]:
# =============== Khởi tạo model ===============
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes)

if torch.cuda.device_count() > 1:
    print("Using DataParallel with", torch.cuda.device_count(), "GPUs")
    model = nn.DataParallel(model)

model.to(device)

In [ ]:
from torchsummary import summary

summary(model, (3, 224, 224))

In [ ]:
# =============== Cấu hình optimizer & loss ===============
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

scaler = torch.amp.GradScaler()

In [ ]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

patience = 5
best_val_acc = 0.0
epochs_no_improve = 0

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_targets, train_preds = [], []

    # Progress bar for training 
    train_loader_iter = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", unit="batch")
    for inputs, targets in train_loader_iter:
        inputs, targets = inputs.cuda(), targets.cuda()
        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * inputs.size(0)
        train_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        train_targets.extend(targets.cpu().numpy())
        train_loader_iter.set_postfix(loss=train_loss/(len(train_preds)), acc=accuracy_score(train_targets, train_preds))

    train_loss /= len(train_loader.dataset)
    train_acc = accuracy_score(train_targets, train_preds)

    # Validation
    model.eval()
    val_loss = 0.0
    val_targets, val_preds = [], []

    val_loader_iter = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]", unit="batch")
    with torch.no_grad():
        for inputs, targets in val_loader_iter:
            inputs, targets = inputs.cuda(), targets.cuda()
            with torch.amp.autocast(device_type='cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            val_loss += loss.item() * inputs.size(0)
            val_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            val_targets.extend(targets.cpu().numpy())

            # chỉ hiển thị loss batch hiện tại
            val_loader_iter.set_postfix(batch_loss=loss.item())

    # Tính metrics sau khi kết thúc toàn bộ validation
    val_loss /= len(val_loader.dataset)
    val_acc = accuracy_score(val_targets, val_preds)
    val_precision = precision_score(val_targets, val_preds, average='macro')
    val_recall = recall_score(val_targets, val_preds, average='macro')
    val_f1 = f1_score(val_targets, val_preds, average='macro')
    conf_matrix = confusion_matrix(val_targets, val_preds)

    print(f"\nEpoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | F1: {val_f1:.4f}")

    scheduler.step()

    # Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

In [ ]:
test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=val_transforms)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

num_classes = len(test_dataset.classes)
print("Số lớp:", num_classes)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from tqdm import tqdm

def evaluate_model(model, test_loader, device='cuda', class_names=None, normalize_cm=True):
    model.to(device)
    model.eval()
    
    all_preds = []
    all_targets = []

    with torch.no_grad():
        test_iter = tqdm(test_loader, desc="Testing", unit="batch")
        for inputs, targets in test_iter:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    # Tự lấy class_names nếu chưa truyền
    if class_names is None:
        class_names = sorted(list(set(all_targets)))

    # Tính metrics
    acc = accuracy_score(all_targets, all_preds)
    precision = precision_score(all_targets, all_preds, average='macro')
    recall = recall_score(all_targets, all_preds, average='macro')
    f1 = f1_score(all_targets, all_preds, average='macro')
    
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")

    # Confusion matrix
    cm = confusion_matrix(all_targets, all_preds)
    if normalize_cm:
        cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
        annot = np.around(cm_percent, 2)
        fmt = '.2f'
    else:
        annot = cm
        fmt = 'd'

    plt.figure(figsize=(10,8))
    sns.heatmap(cm_percent if normalize_cm else cm, annot=annot, fmt=fmt,
                cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

In [ ]:
class_names = test_dataset.classes
print(class_names)

In [ ]:
evaluate_model(model, test_loader, device='cuda', class_names=class_names)

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
evaluate_model(model, test_loader, class_names=class_names)